# Train and Deploy a Breast Cancer Classifier with AWS SageMaker
Third Year B.Tech (AI & DS) end-to-end ML assignment. **Coursework model - not for clinical use.**

The cells are deliberately safe to open or run locally. The AWS operations below require an AWS account and explicit command execution, as they can incur charges.

## 1. Architecture
**Training:** scikit-learn dataset -> stratified split -> private/encrypted S3 -> SageMaker managed training jobs -> `model.tar.gz` in S3.

**Inference:** API Gateway `POST /predict` (AWS_IAM) -> Lambda -> SageMaker real-time endpoint -> predicted benign/malignant class.

**Cross-cutting:** IAM roles, CloudWatch metrics and logs, error alarm, cost safeguards and cleanup.

In [ ]:
from pathlib import Path
from prepare_data import prepare_data
summary = prepare_data(Path('artifacts/data'))
print('Train/test sizes:', summary['train_samples'], summary['test_samples'])

## 2. Reproducible local model baseline
Train both required Random Forest configurations and generate comparison, confusion matrix and ROC. These are local results, **not cloud job evidence**.

In [ ]:
from evaluate_local import run
results = run(Path('artifacts'))
import pandas as pd
display(pd.DataFrame(results)[['n_estimators', 'max_depth', 'accuracy', 'precision', 'recall', 'f1', 'auc', 'malignant_recall', 'false_benign_count']])

In [ ]:
from IPython.display import Image, display
display(Image(filename='artifacts/confusion_matrices.png'))
display(Image(filename='artifacts/roc_curves.png'))

## 3. AWS managed training and real-time endpoint
Execute these terminal commands from this folder after configuring AWS credentials, checking regional quotas and agreeing to current costs. Replace placeholders with your unique identifier. **Do not run from an untrusted notebook**.

```bash
python -m pip install -r requirements-aws.txt
python cloud_pipeline.py provision --bucket ty-aids-sagemaker-ROLLNO --tag rollno --region ap-south-1
python cloud_pipeline.py train
python cloud_pipeline.py deploy
```

Record both AWS jobs' training time, billable seconds, metrics, output S3 location, and `InService` endpoint screenshot.

## 4. Lambda and API Gateway
`lambda_function.py` validates 30 finite numeric features and invokes only the endpoint named by `ENDPOINT_NAME`. API Gateway uses AWS_IAM SigV4 authentication and stage throttling (2 req/s; 5 burst).

```bash
python cloud_pipeline.py publish-api
python cloud_pipeline.py test-api
```

The test saves the genuine API response in `aws_api_test_result.json` for submission.

## 5. Monitoring and resource cleanup
```bash
python cloud_pipeline.py monitor
python cloud_pipeline.py metrics --minutes 15
python cloud_pipeline.py status
# Capture all live resource and CloudWatch screenshots NOW.
python cloud_pipeline.py cleanup --confirm
```

CloudWatch screenshots must show `ModelLatency`, `OverheadLatency`, `Invocations`, `Invocation4XXErrors`, `Invocation5XXErrors` and alarm status. The alarm has no notification destination by default. Capture deletion proof after cleanup. Training/S3/IAM/log resources remain and should be reviewed separately.

## 6. Discussion
**Why false benign is costly:** a malignant tumour classified benign could delay further diagnostic evaluation. The assignment's benign-class recall is not the same as malignant sensitivity; record both. The model is not validated for real clinical decisions.

**Improvements:** assess calibrated probabilities, cost-sensitive learning and malignant-class recall; automate retraining/CI-CD; check dataset shift with SageMaker Model Monitor; add verified alarm notifications and budget controls.

See `README.md` and `screenshots/AWS_SCREENSHOT_CHECKLIST.md` for exact console evidence and teardown steps.